# 05 -- Multi-Agent Supervisor Routing Demo

Implements a simplified, runnable version of the Supervisor/specialist-agent pattern described in
`00-README.md` and Chapter 7: a `Supervisor` class that routes a synthetic commercial loan application
through **mocked** versions of the four specialist agents (Financial Spreading, Risk Assessment, Credit
Policy Compliance, Underwriting Memo Drafting), each represented as a plain Python function standing in
for a real Claude-backed agent.

The point this notebook makes concretely, not just in prose: **the Financial Spreading Agent and Risk
Assessment Agent have no data dependency on each other and are dispatched in parallel** (via
`concurrent.futures`, actually running concurrently and measurably overlapping in wall-clock time,
not just conceptually described as parallel); **the Credit Policy Compliance Agent and Underwriting Memo
Drafting Agent are sequentially dependent** on earlier agents' outputs and are dispatched only once
their dependencies are satisfied. The full dispatch trace -- who ran when, and what they depended on --
is printed and checked with assertions, not just claimed.

Entirely offline -- every "agent" below is a deterministic, seeded mock function that sleeps briefly to
simulate real latency and returns a small structured dict. No real API keys, no network calls, no
external LLM SDKs -- standard library plus `pandas` only.

In [1]:
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field

import pandas as pd

RNG_SEED = 11
random.seed(RNG_SEED)

pd.set_option("display.max_colwidth", 80)
print("Environment ready. Offline, seeded, no network calls.")

Environment ready. Offline, seeded, no network calls.


## 1. A synthetic loan application

Standing in for the documents and application data a real intake step would hand the Supervisor --
enough structured fields for each mocked agent below to have something concrete to work from.

In [2]:
SYNTHETIC_APPLICATION = {
    "application_id": "LOAN-2026-0417",
    "borrower_name": "Meridian Fabrication LLC",
    "industry": "industrial_manufacturing",
    "requested_amount_usd": 2_500_000,
    "financial_statements": {
        "annual_revenue_usd": 9_800_000,
        "total_liabilities_usd": 4_100_000,
        "ebitda_usd": 1_450_000,
        "current_assets_usd": 2_600_000,
        "current_liabilities_usd": 1_300_000,
    },
    "external_risk_signals": {
        "industry_risk_category": "moderate",
        "macro_outlook": "stable",
    },
}

print(f"Loaded synthetic application: {SYNTHETIC_APPLICATION['application_id']} "
      f"({SYNTHETIC_APPLICATION['borrower_name']})")

Loaded synthetic application: LOAN-2026-0417 (Meridian Fabrication LLC)


## 2. The four specialist agents (mocked)

Each function below stands in for a real Claude-backed agent -- a real implementation would call Claude
via Azure AI Foundry, with tool calls to a document-extraction function or Azure AI Search. Here, each
is a small, deterministic mock that sleeps briefly (simulating real network/inference latency) and
returns a structured result.

- `financial_spreading_agent` and `risk_assessment_agent` depend only on the raw application -- **no
  dependency on each other or on any other agent's output**.
- `credit_policy_compliance_agent` depends on `financial_spreading_agent`'s ratios.
- `underwriting_memo_drafting_agent` depends on all three other agents' outputs.

In [3]:
AGENT_LATENCY_SECONDS = {
    "financial_spreading": 0.5,
    "risk_assessment": 0.4,
    "credit_policy_compliance": 0.3,
    "underwriting_memo_drafting": 0.3,
}


def financial_spreading_agent(application):
    """No dependency on any other agent -- reads only the raw application."""
    time.sleep(AGENT_LATENCY_SECONDS["financial_spreading"])
    stmts = application["financial_statements"]
    current_ratio = round(stmts["current_assets_usd"] / stmts["current_liabilities_usd"], 2)
    leverage_ratio = round(stmts["total_liabilities_usd"] / stmts["ebitda_usd"], 2)
    return {
        "agent": "financial_spreading",
        "current_ratio": current_ratio,
        "leverage_ratio": leverage_ratio,
        "ebitda_usd": stmts["ebitda_usd"],
    }


def risk_assessment_agent(application):
    """No dependency on any other agent -- reads only the raw application's external risk signals."""
    time.sleep(AGENT_LATENCY_SECONDS["risk_assessment"])
    signals = application["external_risk_signals"]
    risk_score = {"low": 20, "moderate": 45, "elevated": 70, "high": 90}.get(
        signals["industry_risk_category"], 50
    )
    return {
        "agent": "risk_assessment",
        "risk_score": risk_score,
        "industry_risk_category": signals["industry_risk_category"],
        "macro_outlook": signals["macro_outlook"],
    }


def credit_policy_compliance_agent(application, financial_result):
    """Depends on financial_spreading_agent's ratios -- cannot run before that agent completes."""
    if financial_result is None:
        raise ValueError("credit_policy_compliance_agent requires financial_spreading_agent's output")
    time.sleep(AGENT_LATENCY_SECONDS["credit_policy_compliance"])
    # Toy policy: current ratio must be >= 1.2, leverage ratio must be <= 4.0
    current_ratio_ok = financial_result["current_ratio"] >= 1.2
    leverage_ratio_ok = financial_result["leverage_ratio"] <= 4.0
    return {
        "agent": "credit_policy_compliance",
        "current_ratio_ok": current_ratio_ok,
        "leverage_ratio_ok": leverage_ratio_ok,
        "policy_citations": (
            ["Policy 4.2: minimum current ratio 1.2", "Policy 4.5: maximum leverage ratio 4.0"]
        ),
    }


def underwriting_memo_drafting_agent(application, financial_result, risk_result, compliance_result):
    """Depends on all three other agents' outputs -- the final synthesis step."""
    missing = [
        name
        for name, result in [
            ("financial_spreading", financial_result),
            ("risk_assessment", risk_result),
            ("credit_policy_compliance", compliance_result),
        ]
        if result is None
    ]
    if missing:
        raise ValueError(f"underwriting_memo_drafting_agent missing upstream results: {missing}")
    time.sleep(AGENT_LATENCY_SECONDS["underwriting_memo_drafting"])
    compliant = compliance_result["current_ratio_ok"] and compliance_result["leverage_ratio_ok"]
    recommendation = "APPROVE_WITH_STANDARD_TERMS" if compliant else "ESCALATE_TO_HUMAN_REVIEW"
    return {
        "agent": "underwriting_memo_drafting",
        "borrower": application["borrower_name"],
        "recommendation": recommendation,
        "cites_financial_ratios": True,
        "cites_risk_score": True,
        "cites_policy_checks": True,
    }


print("Four specialist agent functions defined.")

Four specialist agent functions defined.


## 3. The Supervisor

Routes the application through the four agents in the correct order, running the two independent agents
**concurrently** via `ThreadPoolExecutor` and the two dependent agents **sequentially**, once their
dependencies are satisfied. Every dispatch is timestamped (relative to the run's start) so the trace can
be inspected and the parallelism verified, not just assumed.

In [4]:
@dataclass
class Supervisor:
    trace: list = field(default_factory=list)

    def _record(self, agent_name, start, end, depends_on):
        self.trace.append(
            {
                "agent": agent_name,
                "start_offset_s": round(start, 3),
                "end_offset_s": round(end, 3),
                "duration_s": round(end - start, 3),
                "depends_on": ", ".join(depends_on) if depends_on else "(none -- can run immediately)",
            }
        )

    def run(self, application):
        run_start = time.monotonic()
        self.trace.clear()

        # --- Phase 1: dispatch the two independent agents IN PARALLEL. ---
        # Neither financial_spreading_agent nor risk_assessment_agent depends on the other, or on
        # anything except the raw application -- this is exactly the pair Chapter 7 argues should run
        # concurrently to recover wall-clock latency, since paying for both calls is unavoidable but
        # paying for both calls IN SERIES is not.
        results = {}
        with ThreadPoolExecutor(max_workers=2) as pool:
            futures = {
                pool.submit(financial_spreading_agent, application): "financial_spreading",
                pool.submit(risk_assessment_agent, application): "risk_assessment",
            }
            starts = {name: time.monotonic() - run_start for name in futures.values()}
            for future in as_completed(futures):
                name = futures[future]
                result = future.result()
                end = time.monotonic() - run_start
                self._record(name, starts[name], end, depends_on=[])
                results[name] = result

        # --- Phase 2: dispatch the compliance agent SEQUENTIALLY -- it needs financial_spreading's
        # output, which is only guaranteed available once Phase 1 has fully completed. ---
        start = time.monotonic() - run_start
        compliance_result = credit_policy_compliance_agent(application, results["financial_spreading"])
        end = time.monotonic() - run_start
        self._record("credit_policy_compliance", start, end, depends_on=["financial_spreading"])
        results["credit_policy_compliance"] = compliance_result

        # --- Phase 3: dispatch the memo drafting agent SEQUENTIALLY -- it needs all three other
        # agents' outputs, so it can only run after every earlier phase has completed. ---
        start = time.monotonic() - run_start
        memo_result = underwriting_memo_drafting_agent(
            application,
            results["financial_spreading"],
            results["risk_assessment"],
            results["credit_policy_compliance"],
        )
        end = time.monotonic() - run_start
        self._record(
            "underwriting_memo_drafting",
            start,
            end,
            depends_on=["financial_spreading", "risk_assessment", "credit_policy_compliance"],
        )
        results["underwriting_memo_drafting"] = memo_result

        total_elapsed = time.monotonic() - run_start
        return results, total_elapsed


print("Supervisor defined.")

Supervisor defined.


## 4. Run the Supervisor against the synthetic application

In [5]:
supervisor = Supervisor()
results, total_elapsed = supervisor.run(SYNTHETIC_APPLICATION)

trace_df = pd.DataFrame(supervisor.trace)
print(f"Total wall-clock time for the full run: {total_elapsed:.3f}s\n")
trace_df

Total wall-clock time for the full run: 1.102s



,agent,start_offset_s,end_offset_s,duration_s,depends_on
0,risk_assessment,0.001,0.401,0.401,(none -- can run immediately)
1,financial_spreading,0.001,0.501,0.500,(none -- can run immediately)
2,credit_policy_compliance,0.501,0.802,0.300,financial_spreading
3,underwriting_memo_drafting,0.802,1.102,0.300,"financial_spreading, risk_assessment, credit_policy_compliance"


## 5. Verify the parallel pair actually overlapped in wall-clock time

Not just "we called them both" -- the assertions below check that the Financial Spreading Agent's and
Risk Assessment Agent's execution windows genuinely overlapped, and that the total time for that phase
was closer to the *slower* of the two agents than to their *sum* -- the concrete, measurable benefit of
dispatching them concurrently instead of one after the other.

In [6]:
fin_row = trace_df[trace_df["agent"] == "financial_spreading"].iloc[0]
risk_row = trace_df[trace_df["agent"] == "risk_assessment"].iloc[0]

# Overlap check: each agent's window must start before the other one ends.
overlap = (fin_row["start_offset_s"] < risk_row["end_offset_s"]) and (
    risk_row["start_offset_s"] < fin_row["end_offset_s"]
)
assert overlap, "Expected financial_spreading and risk_assessment to run concurrently, not in series."
print(f"PASS: financial_spreading [{fin_row['start_offset_s']:.3f}s -> {fin_row['end_offset_s']:.3f}s] "
      f"and risk_assessment [{risk_row['start_offset_s']:.3f}s -> {risk_row['end_offset_s']:.3f}s] "
      f"overlap in wall-clock time.")

parallel_phase_elapsed = max(fin_row["end_offset_s"], risk_row["end_offset_s"])
serial_equivalent = AGENT_LATENCY_SECONDS["financial_spreading"] + AGENT_LATENCY_SECONDS["risk_assessment"]
print(f"\nParallel phase wall-clock time: {parallel_phase_elapsed:.3f}s")
print(f"What that same phase would have cost running in series: ~{serial_equivalent:.3f}s")
assert parallel_phase_elapsed < serial_equivalent, (
    "Expected the parallel dispatch to be faster than the serial equivalent."
)
print("PASS: parallel dispatch measurably recovered wall-clock time versus running the two "
      "independent agents in series -- the concrete benefit Chapter 7 argues for.")

PASS: financial_spreading [0.001s -> 0.501s] and risk_assessment [0.001s -> 0.401s] overlap in wall-clock time.

Parallel phase wall-clock time: 0.501s
What that same phase would have cost running in series: ~0.900s
PASS: parallel dispatch measurably recovered wall-clock time versus running the two independent agents in series -- the concrete benefit Chapter 7 argues for.


## 6. Verify the sequential pair only ran after their dependencies were satisfied

In [7]:
compliance_row = trace_df[trace_df["agent"] == "credit_policy_compliance"].iloc[0]
memo_row = trace_df[trace_df["agent"] == "underwriting_memo_drafting"].iloc[0]

assert compliance_row["start_offset_s"] >= fin_row["end_offset_s"], (
    "credit_policy_compliance must not start before financial_spreading (its dependency) completes."
)
print("PASS: credit_policy_compliance started only after financial_spreading completed "
      f"({compliance_row['start_offset_s']:.3f}s >= {fin_row['end_offset_s']:.3f}s).")

latest_dependency_end = max(
    fin_row["end_offset_s"], risk_row["end_offset_s"], compliance_row["end_offset_s"]
)
assert memo_row["start_offset_s"] >= latest_dependency_end, (
    "underwriting_memo_drafting must not start before ALL three upstream agents complete."
)
print("PASS: underwriting_memo_drafting started only after all three upstream agents completed "
      f"({memo_row['start_offset_s']:.3f}s >= {latest_dependency_end:.3f}s).")

print("\nFinal recommendation:", results["underwriting_memo_drafting"]["recommendation"])

PASS: credit_policy_compliance started only after financial_spreading completed (0.501s >= 0.501s).
PASS: underwriting_memo_drafting started only after all three upstream agents completed (0.802s >= 0.802s).

Final recommendation: APPROVE_WITH_STANDARD_TERMS


## Recap

This notebook implemented, end to end, the routing shape `00-README.md` and Chapter 7 describe in
prose: a Supervisor that dispatches the two data-independent agents (Financial Spreading, Risk
Assessment) **concurrently**, verified here by measuring real wall-clock overlap rather than assuming
it, and dispatches the two dependent agents (Credit Policy Compliance, Underwriting Memo Drafting)
**sequentially**, only once their actual dependencies are satisfied -- also verified against the trace,
not just asserted in prose. A real production Supervisor built on LangGraph (course 8's LangGraph
fundamentals chapter, applied to this specific five-agent roster) expresses this same dependency graph
as a state graph rather than the linear Python control flow used here for clarity, but the underlying
shape -- run what doesn't depend on anything else in parallel, run what does depend on something else
only once that something is ready -- is identical.